## 1. Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_PATH    = '/content/drive/MyDrive/echonet/EchoNet-Dynamic.zip'
EXTRACT_DIR = '/content/echonet'

if not os.path.exists(EXTRACT_DIR):
    print('Unzipping dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Dataset already extracted.')

VIDEO_DIR = os.path.join(EXTRACT_DIR, 'EchoNet-Dynamic', 'Videos')
if not os.path.exists(VIDEO_DIR):
    VIDEO_DIR = os.path.join(EXTRACT_DIR, 'Videos')
print(f'Video dir: {VIDEO_DIR}')


Mounted at /content/drive
Unzipping dataset...
Done.
Video dir: /content/echonet/EchoNet-Dynamic/Videos


## 2. Imports

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import time

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 3. Configuration

In [ ]:
FRAMES_PER_CLIP = 100          # number of frames per clip
MAX_FRAMES      = 400          # only read first 400 frames of each video
TRAIN_SPLIT     = 300          # first 300 videos for training
CACHE_DIR       = '/content/flow_stats_cache'   # local Colab disk, fast
CSV_PATH        = '/content/drive/MyDrive/heart_rates.csv'
OUTPUT_DIR      = '/content/drive/MyDrive/echonet_project/mlr_flow_stats'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Flow stats cache → {CACHE_DIR}')
print(f'Outputs          → {OUTPUT_DIR}')


Flow stats cache → /content/flow_stats_cache
Outputs          → /content/drive/MyDrive/echonet_project/mlr_flow_stats


## 4. Pre-compute & Cache Flow Statistics

For each video we compute Farneback optical flow between every pair of
consecutive frames (up to `MAX_FRAMES`).  For each flow field we compute the
**per-pixel magnitude** `sqrt(u² + v²)`, then collect all pixel magnitudes
across all flow fields in a clip of `FRAMES_PER_CLIP` frames.

From that pooled set of magnitudes we derive the six scalar features used as
inputs to the linear regression:

| Feature | Description |
|---------|-------------|
| `mean_mag`  | mean pixel-level flow magnitude across all frames |
| `p90_mag`   | 90th percentile |
| `p95_mag`   | 95th percentile |
| `p99_mag`   | 99th percentile |
| `p999_mag`  | 99.9th percentile |
| `std_mag`   | standard deviation |

Each video's statistics are cached as a single `.npz` file so the expensive
optical-flow computation only runs once per session.


In [ ]:
def cache_path_for(stem, cache_dir):
    return Path(cache_dir) / f'{stem}.npz'


def compute_and_cache_flow_stats(video_path, cache_dir,
                                 target_size=(112, 112),
                                 max_frames=MAX_FRAMES,
                                 clip_len=FRAMES_PER_CLIP):
    """
    Compute Farneback optical flow for consecutive frame pairs (up to
    max_frames), then derive per-clip flow-magnitude statistics.

    For each non-overlapping clip of clip_len frames we pool all pixel
    magnitudes across all (clip_len - 1) flow fields and compute:
        mean, p90, p95, p99, p99.9, std

    Results are saved as a .npz file with arrays of shape (n_clips,).
    Returns the cache path.
    """
    stem       = Path(video_path).stem
    out_path   = cache_path_for(stem, cache_dir)
    if out_path.exists():
        return str(out_path)

    # ── Read frames ──────────────────────────────────────────────────────────
    cap    = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, target_size)
        frames.append(gray)
    cap.release()

    if len(frames) < 2:
        # Save empty arrays so the rest of the pipeline can skip cleanly
        np.savez(str(out_path),
                 mean_mag=np.array([]), p90_mag=np.array([]),
                 p95_mag=np.array([]),  p99_mag=np.array([]),
                 p999_mag=np.array([]), std_mag=np.array([]))
        return str(out_path)

    # ── Compute all flow fields → magnitudes ─────────────────────────────────
    all_mags = []   # list of (H*W,) arrays, one per flow field
    for i in range(len(frames) - 1):
        flow = cv2.calcOpticalFlowFarneback(
            frames[i], frames[i + 1], None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )
        mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2).ravel()
        all_mags.append(mag.astype(np.float32))

    # ── Slice into non-overlapping clips of (clip_len - 1) flow fields ───────
    clip_flows  = clip_len - 1
    n_flows     = len(all_mags)

    # Build clip intervals (same logic as the ResNet notebook)
    intervals = []
    start = 0
    while start + clip_flows <= n_flows:
        intervals.append((start, start + clip_flows))
        start += clip_flows
    if not intervals:
        intervals = [(0, n_flows)]   # shorter-than-one-clip video
    elif intervals[-1][1] < n_flows:
        intervals.append((n_flows - clip_flows, n_flows))

    stats = {k: [] for k in ('mean_mag', 'p90_mag', 'p95_mag',
                              'p99_mag', 'p999_mag', 'std_mag')}

    for fs, fe in intervals:
        clip_mags = np.concatenate(all_mags[fs:fe])   # all pixels, all frames
        stats['mean_mag'].append(float(clip_mags.mean()))
        stats['p90_mag'].append(float(np.percentile(clip_mags, 90)))
        stats['p95_mag'].append(float(np.percentile(clip_mags, 95)))
        stats['p99_mag'].append(float(np.percentile(clip_mags, 99)))
        stats['p999_mag'].append(float(np.percentile(clip_mags, 99.9)))
        stats['std_mag'].append(float(clip_mags.std()))

    np.savez(str(out_path), **{k: np.array(v) for k, v in stats.items()})
    return str(out_path)


def build_cache(df, video_dir, cache_dir):
    """Pre-compute and cache flow statistics for every video in df."""
    t0 = time.time()
    paths = []
    for _, row in df.iterrows():
        fname = row['FileName']
        p = Path(video_dir) / f'{fname}.avi'
        if not p.exists():
            p = Path(video_dir) / fname
        if p.exists():
            paths.append(str(p))

    print(f'Caching flow stats for {len(paths)} videos...')
    for i, p in enumerate(paths):
        compute_and_cache_flow_stats(p, cache_dir)
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(paths)}  ({time.time()-t0:.0f}s)')
    print(f'Done in {time.time()-t0:.0f}s')


## 5. Build Feature Matrix

In [ ]:
FEATURE_NAMES = ['mean_mag', 'p90_mag', 'p95_mag', 'p99_mag', 'p999_mag', 'std_mag']


def build_feature_matrix(df, video_dir, cache_dir):
    """
    Load cached statistics for each video and expand to one row per clip.
    Returns X (n_clips, 6) and y (n_clips,).
    """
    rows, labels = [], []
    skipped = 0

    for _, row in df.iterrows():
        fname = row['FileName']
        hr    = float(row['heart_rate'])

        p = Path(video_dir) / f'{fname}.avi'
        if not p.exists():
            p = Path(video_dir) / fname
        if not p.exists():
            skipped += 1
            continue

        cp = cache_path_for(p.stem, cache_dir)
        if not cp.exists():
            skipped += 1
            continue

        data = np.load(str(cp))
        n_clips = len(data['mean_mag'])
        if n_clips == 0:
            skipped += 1
            continue

        for i in range(n_clips):
            row_feats = [float(data[feat][i]) for feat in FEATURE_NAMES]
            rows.append(row_feats)
            labels.append(hr)

    if skipped:
        print(f'Skipped {skipped} videos (missing file or cache)')

    X = np.array(rows, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)
    print(f'Feature matrix: {X.shape}  |  Labels: {y.shape}')
    return X, y


## 6. Load CSV, Build Cache & Feature Matrices

In [ ]:
df       = pd.read_csv(CSV_PATH)
train_df = df.iloc[:TRAIN_SPLIT].reset_index(drop=True)
val_df   = df.iloc[TRAIN_SPLIT:].reset_index(drop=True)

print(f'Train videos: {len(train_df)} | Val videos: {len(val_df)}')

# Pre-compute and cache all flow statistics (done once per session)
build_cache(pd.concat([train_df, val_df], ignore_index=True), VIDEO_DIR, CACHE_DIR)

print('\nBuilding training feature matrix...')
X_train, y_train = build_feature_matrix(train_df, VIDEO_DIR, CACHE_DIR)
print('Building validation feature matrix...')
X_val,   y_val   = build_feature_matrix(val_df,   VIDEO_DIR, CACHE_DIR)


Train videos: 300 | Val videos: 72
Caching flow stats for 372 videos...
  50/372  (45s)
  100/372  (89s)
  150/372  (131s)
  200/372  (173s)
  250/372  (215s)
  300/372  (258s)
  350/372  (301s)
Done in 319s

Building training feature matrix...
Feature matrix: (679, 6)  |  Labels: (679,)
Building validation feature matrix...
Feature matrix: (159, 6)  |  Labels: (159,)


## 7. Train Multiple Linear Regression

Ordinary least squares regression:

    HR = β₀ + β₁·mean_mag + β₂·p90_mag + β₃·p95_mag
             + β₄·p99_mag  + β₅·p999_mag + β₆·std_mag

No regularisation is applied — we want interpretable coefficients that show
the raw contribution of each flow-magnitude statistic to the heart-rate
prediction.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('=' * 60)
print('FITTED EQUATION')
print('=' * 60)
print(f'  Intercept (β₀) : {model.intercept_:+.4f}')
for name, coef in zip(FEATURE_NAMES, model.coef_):
    print(f'  {name:<12s} (β)  : {coef:+.4f}')
print('=' * 60)
print()
print('Full equation:')
terms = ' '.join(f'{coef:+.4f}·{name}' for name, coef in zip(FEATURE_NAMES, model.coef_))
print(f'  HR = {model.intercept_:.4f} {terms}')


FITTED EQUATION
  Intercept (β₀) : +48.4772
  mean_mag     (β)  : +680.2460
  p90_mag      (β)  : -381.2529
  p95_mag      (β)  : -16.5016
  p99_mag      (β)  : -46.3693
  p999_mag     (β)  : -26.3989
  std_mag      (β)  : +723.9584

Full equation:
  HR = 48.4772 +680.2460·mean_mag -381.2529·p90_mag -16.5016·p95_mag -46.3693·p99_mag -26.3989·p999_mag +723.9584·std_mag


## 8. Evaluate on Validation Set

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Clip-level predictions ────────────────────────────────────────────────────
y_pred_clips = model.predict(X_val)

clip_mae  = mean_absolute_error(y_val, y_pred_clips)
clip_rmse = float(np.sqrt(mean_squared_error(y_val, y_pred_clips)))
clip_mape = float(np.mean(np.abs((y_pred_clips - y_val) / (y_val + 1e-6)) * 100))
clip_p90  = float(np.percentile(np.abs(y_pred_clips - y_val), 90))
clip_med  = float(np.median(np.abs(y_pred_clips - y_val)))

# ── Video-level predictions (average clips per video) ────────────────────────
# We need to know which clips belong to which video.
# Rebuild the mapping by iterating val_df in the same order as build_feature_matrix.
video_preds  = {}   # stem -> list of clip predictions
video_labels = {}   # stem -> true HR

clip_idx = 0
for _, row in val_df.iterrows():
    fname = row['FileName']
    hr    = float(row['heart_rate'])

    p = Path(VIDEO_DIR) / f'{fname}.avi'
    if not p.exists():
        p = Path(VIDEO_DIR) / fname
    if not p.exists():
        continue

    cp = cache_path_for(p.stem, CACHE_DIR)
    if not cp.exists():
        continue

    data    = np.load(str(cp))
    n_clips = len(data['mean_mag'])
    if n_clips == 0:
        continue

    clip_preds_for_video = y_pred_clips[clip_idx : clip_idx + n_clips].tolist()
    video_preds[p.stem]  = clip_preds_for_video
    video_labels[p.stem] = hr
    clip_idx += n_clips

# Aggregate
records = []
for stem, preds in video_preds.items():
    true_bpm = video_labels[stem]
    est_bpm  = float(np.mean(preds))
    records.append({
        'video'   : stem,
        'true_bpm': true_bpm,
        'est_bpm' : est_bpm,
        'abs_err' : abs(est_bpm - true_bpm),
        'pct_err' : abs(est_bpm - true_bpm) / true_bpm * 100,
    })

res_df     = pd.DataFrame(records)
true_bpm_v = res_df['true_bpm'].values
est_bpm_v  = res_df['est_bpm'].values
abs_err_v  = res_df['abs_err'].values
diff_v     = est_bpm_v - true_bpm_v
mean_val_v = (true_bpm_v + est_bpm_v) / 2
bias       = diff_v.mean()
sd         = diff_v.std()

vid_mae    = abs_err_v.mean()
vid_median = float(np.median(abs_err_v))
vid_p90    = float(np.percentile(abs_err_v, 90))
vid_mape   = res_df['pct_err'].mean()
vid_rmse   = float(np.sqrt((abs_err_v ** 2).mean()))
n          = len(res_df)

print('\n' + '=' * 55)
print('VALIDATION RESULTS — Multiple Linear Regression')
print('=' * 55)
print(f'  N                     : {n}')
print()
print('  — Video-level (clips averaged per video) —')
print(f'  MAE        : {vid_mae:.2f} BPM')
print(f'  Median AE  : {vid_median:.2f} BPM')
print(f'  P90 AE     : {vid_p90:.2f} BPM')
print(f'  MAPE       : {vid_mape:.2f}%')
print(f'  RMSE       : {vid_rmse:.2f} BPM')
print()
print('  — Clip-level —')
print(f'  MAE        : {clip_mae:.2f} BPM')
print(f'  Median AE  : {clip_med:.2f} BPM')
print(f'  P90 AE     : {clip_p90:.2f} BPM')
print(f'  MAPE       : {clip_mape:.2f}%')
print(f'  RMSE       : {clip_rmse:.2f} BPM')
print('=' * 55)

summary = pd.DataFrame([{
    'Method'          : 'Multiple Linear Regression (flow stats)',
    'N (videos)'      : n,
    'MAE (BPM)'       : vid_mae,
    'Median AE (BPM)' : vid_median,
    'P90 AE (BPM)'    : vid_p90,
    'MAPE (%)'        : vid_mape,
    'RMSE (BPM)'      : vid_rmse,
    'Clip MAE (BPM)'  : clip_mae,
    'Clip RMSE (BPM)' : clip_rmse,
}]).set_index('Method')
summary.to_csv(Path(OUTPUT_DIR) / 'val_metrics_summary.csv')
print(f'Metrics saved → {OUTPUT_DIR}/val_metrics_summary.csv')



VALIDATION RESULTS — Multiple Linear Regression
  N                     : 72

  — Video-level (clips averaged per video) —
  MAE        : 10.07 BPM
  Median AE  : 7.96 BPM
  P90 AE     : 21.23 BPM
  MAPE       : 14.69%
  RMSE       : 12.79 BPM

  — Clip-level —
  MAE        : 10.47 BPM
  Median AE  : 8.30 BPM
  P90 AE     : 22.97 BPM
  MAPE       : 15.35%
  RMSE       : 13.26 BPM
Metrics saved → /content/drive/MyDrive/echonet_project/mlr_flow_stats/val_metrics_summary.csv


## 9. Plots

In [ ]:
VAL_PLOTS = Path(OUTPUT_DIR) / 'val_plots'
VAL_PLOTS.mkdir(parents=True, exist_ok=True)

# ── Bland-Altman (video-level) ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(mean_val_v, diff_v, alpha=0.6, s=30, color='steelblue')
ax.axhline(bias,              color='black', linewidth=1.5, label=f'Bias: {bias:+.1f}')
ax.axhline(bias + 1.96 * sd, color='gray',  linestyle='--',
           label=f'+1.96 SD: {bias+1.96*sd:+.1f}')
ax.axhline(bias - 1.96 * sd, color='gray',  linestyle='--',
           label=f'-1.96 SD: {bias-1.96*sd:+.1f}')
ax.axhline(0, color='green', linewidth=0.8, linestyle=':', alpha=0.5)
ax.set_xlabel('Mean of True & Estimated BPM')
ax.set_ylabel('Estimated − True (BPM)')
ax.set_title('Bland-Altman — Multiple Linear Regression (video-level)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(VAL_PLOTS / 'bland_altman.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {VAL_PLOTS}/bland_altman.png')

# ── Scatter: estimated vs true ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(true_bpm_v, est_bpm_v, alpha=0.5, s=30, color='steelblue')
lo, hi = true_bpm_v.min() - 5, true_bpm_v.max() + 5
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect')
ax.set_xlabel('True BPM')
ax.set_ylabel('Estimated BPM')
ax.set_title('Estimated vs True BPM — Multiple Linear Regression')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(VAL_PLOTS / 'scatter_est_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {VAL_PLOTS}/scatter_est_vs_true.png')

# ── Absolute error distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(abs_err_v, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(vid_mae,    color='black', linestyle='-',  linewidth=1.5, label=f'MAE {vid_mae:.1f}')
ax.axvline(vid_median, color='orange', linestyle='--', linewidth=1.5, label=f'Median {vid_median:.1f}')
ax.axvline(vid_p90,    color='red',   linestyle=':',  linewidth=1.5, label=f'P90 {vid_p90:.1f}')
ax.set_xlabel('Absolute Error (BPM)')
ax.set_ylabel('Count')
ax.set_title('Absolute Error Distribution — Multiple Linear Regression')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(VAL_PLOTS / 'error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {VAL_PLOTS}/error_distribution.png')

# ── Coefficient bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if c >= 0 else 'tomato' for c in model.coef_]
ax.bar(FEATURE_NAMES, model.coef_, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
ax.set_title('Linear Regression Coefficients')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(VAL_PLOTS / 'coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {VAL_PLOTS}/coefficients.png')


Saved → /content/drive/MyDrive/echonet_project/mlr_flow_stats/val_plots/bland_altman.png
Saved → /content/drive/MyDrive/echonet_project/mlr_flow_stats/val_plots/scatter_est_vs_true.png
Saved → /content/drive/MyDrive/echonet_project/mlr_flow_stats/val_plots/error_distribution.png
Saved → /content/drive/MyDrive/echonet_project/mlr_flow_stats/val_plots/coefficients.png


## 10. Final Summary

In [ ]:
print('\n' + '=' * 60)
print('FINAL SUMMARY — MULTIPLE LINEAR REGRESSION (flow statistics)')
print('=' * 60)
print()
print('Fitted equation:')
print(f'  HR = {model.intercept_:.4f}')
for name, coef in zip(FEATURE_NAMES, model.coef_):
    print(f'       {coef:+.4f} * {name}')
print()
print('Validation performance (video-level, clips averaged):')
print(f'  N          : {n}')
print(f'  MAE        : {vid_mae:.2f} BPM')
print(f'  Median AE  : {vid_median:.2f} BPM')
print(f'  P90 AE     : {vid_p90:.2f} BPM')
print(f'  MAPE       : {vid_mape:.2f}%')
print(f'  RMSE       : {vid_rmse:.2f} BPM')
print()
print(f'All outputs saved to: {OUTPUT_DIR}')
print('=' * 60)



FINAL SUMMARY — MULTIPLE LINEAR REGRESSION (flow statistics)

Fitted equation:
  HR = 48.4772
       +680.2460 * mean_mag
       -381.2529 * p90_mag
       -16.5016 * p95_mag
       -46.3693 * p99_mag
       -26.3989 * p999_mag
       +723.9584 * std_mag

Validation performance (video-level, clips averaged):
  N          : 72
  MAE        : 10.07 BPM
  Median AE  : 7.96 BPM
  P90 AE     : 21.23 BPM
  MAPE       : 14.69%
  RMSE       : 12.79 BPM

All outputs saved to: /content/drive/MyDrive/echonet_project/mlr_flow_stats


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── 1. Standardised coefficients ─────────────────────────────────────────────
# Z-score each feature using training set statistics, then refit
scaler = StandardScaler()
X_train_z = scaler.fit_transform(X_train)
X_val_z   = scaler.transform(X_val)

model_std = LinearRegression()
model_std.fit(X_train_z, y_train)

std_coefs = pd.Series(model_std.coef_, index=FEATURE_NAMES).sort_values(key=abs, ascending=False)

print('=' * 60)
print('STANDARDISED COEFFICIENTS')
print('(BPM change per 1 std-dev increase in that feature)')
print('=' * 60)
for name, coef in std_coefs.items():
    bar = '█' * int(abs(coef) / std_coefs.abs().max() * 30)
    sign = '+' if coef >= 0 else '-'
    print(f'  {name:<12s}  {coef:+7.3f}  {sign}{bar}')
print('=' * 60)

# ── 2. Pearson correlations ───────────────────────────────────────────────────
correlations = pd.Series(
    [float(np.corrcoef(X_train[:, i], y_train)[0, 1]) for i in range(X_train.shape[1])],
    index=FEATURE_NAMES
).sort_values(key=abs, ascending=False)

print()
print('=' * 60)
print('PEARSON CORRELATIONS WITH HEART RATE')
print('(ignores other features; +/- 1.0 = perfect linear relationship)')
print('=' * 60)
for name, r in correlations.items():
    bar = '█' * int(abs(r) * 30)
    sign = '+' if r >= 0 else '-'
    print(f'  {name:<12s}  {r:+6.3f}  {sign}{bar}')
print('=' * 60)

# ── 3. Variance Inflation Factor ──────────────────────────────────────────────
vifs = pd.Series(
    [variance_inflation_factor(X_train_z, i) for i in range(X_train_z.shape[1])],
    index=FEATURE_NAMES
).sort_values(ascending=False)

print()
print('=' * 60)
print('VARIANCE INFLATION FACTORS')
print('(VIF > 10 = problematic multicollinearity;')
print(' VIF > 100 = coefficient is essentially meaningless)')
print('=' * 60)
for name, vif in vifs.items():
    flag = '  *** HIGH ***' if vif > 10 else ''
    print(f'  {name:<12s}  {vif:8.1f}{flag}')
print('=' * 60)

# ── 4. Feature correlation matrix (shows which features are collinear) ────────
corr_matrix = np.corrcoef(X_train.T)
print()
print('=' * 60)
print('INTER-FEATURE CORRELATION MATRIX')
print('(values near +/-1.0 mean those two features are near-redundant)')
print('=' * 60)
corr_df = pd.DataFrame(corr_matrix, index=FEATURE_NAMES, columns=FEATURE_NAMES)
pd.set_option('display.float_format', '{:+.3f}'.format)
pd.set_option('display.width', 120)
print(corr_df.to_string())
print('=' * 60)

# ── 5. Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Standardised coefficients
ax = axes[0]
colors = ['steelblue' if c >= 0 else 'tomato' for c in std_coefs.values]
ax.barh(std_coefs.index, std_coefs.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised coefficient (BPM / 1 std-dev)')
ax.set_title('Standardised Coefficients\n(comparable importance)')
ax.grid(True, axis='x', alpha=0.3)

# Pearson correlations
ax = axes[1]
colors = ['steelblue' if r >= 0 else 'tomato' for r in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlim(-1, 1)
ax.set_xlabel('Pearson r with heart rate')
ax.set_title('Pearson Correlations\n(each feature vs HR independently)')
ax.grid(True, axis='x', alpha=0.3)

# VIF
ax = axes[2]
colors = ['tomato' if v > 10 else 'steelblue' for v in vifs.values]
ax.barh(vifs.index, vifs.values, color=colors, edgecolor='white')
ax.axvline(10,  color='orange', linewidth=1.2, linestyle='--', label='VIF = 10')
ax.axvline(100, color='red',    linewidth=1.2, linestyle='--', label='VIF = 100')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor\n(red = multicollinearity problem)')
ax.legend(fontsize=8)
ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Feature Importance Analysis — MLR Flow Statistics', fontweight='bold')
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'val_plots' / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {OUTPUT_DIR}/val_plots/feature_importance.png")

STANDARDISED COEFFICIENTS
(BPM change per 1 std-dev increase in that feature)
  p90_mag       -53.728  -██████████████████████████████
  std_mag       +48.617  +███████████████████████████
  mean_mag      +41.250  +███████████████████████
  p999_mag      -17.233  -█████████
  p99_mag       -14.297  -███████
  p95_mag        -2.994  -█

PEARSON CORRELATIONS WITH HEART RATE
(ignores other features; +/- 1.0 = perfect linear relationship)
  mean_mag      +0.395  +███████████
  p90_mag       +0.284  +████████
  p95_mag       +0.224  +██████
  std_mag       +0.174  +█████
  p99_mag       +0.147  +████
  p999_mag      +0.054  +█

VARIANCE INFLATION FACTORS
(VIF > 10 = problematic multicollinearity;
 VIF > 100 = coefficient is essentially meaningless)
  std_mag          722.5  *** HIGH ***
  p90_mag          428.4  *** HIGH ***
  p95_mag          213.3  *** HIGH ***
  p99_mag          114.6  *** HIGH ***
  mean_mag          55.2  *** HIGH ***
  p999_mag          47.8  *** HIGH ***

INTER-FEATU